## Step 1: Import Keras and Keras Tuner
- Purpose: load the deep learning library and tuning toolkit.
- Why: Keras builds the model; Keras Tuner searches hyperparameters.
- Expected output: no printed output if imports succeed.

In [ ]:
import keras
import keras_tuner

## Step 2: Define a model-building function
- Purpose: create a model whose hyperparameters can be tuned.
- Why: Keras Tuner calls this function to build models with different settings.
- Expected output: a compiled model for each hyperparameter trial.

In [ ]:
def build_model(hp):
    model = keras.Sequential()
    model.add(keras.layers.Dense(
        units=hp.Int('units', min_value=8, max_value=128, step=8),
        activation='relu'))
    model.add(keras.layers.Dense(1, activation='linear'))
    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Float('learning_rate', 1e-4, 1e-2, sampling='log')),
        loss='mse',
        metrics=['mae'])
    return model

## Step 3: Configure the tuner
- Purpose: set search strategy and objective for hyperparameter tuning.
- Why: RandomSearch explores combinations of units and learning rates.
- Expected output: a `tuner` object ready to search.

In [ ]:
tuner = keras_tuner.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=10,
    executions_per_trial=1,
    directory='my_dir',
    project_name='keras_tuner_demo')

## Step 4: Run the hyperparameter search
- Purpose: train multiple models with different hyperparameters.
- Why: identify the configuration that minimizes validation loss.
- Expected output: search logs for each trial.
- Note: ensure `x_train` and `y_train` are defined and preprocessed before this step.

In [ ]:
tuner.search(x_train, y_train, epochs=10, validation_split=0.2)


Trial 10 Complete [00h 00m 03s]
val_loss: 0.06770023703575134

Best val_loss So Far: 0.06770023703575134
Total elapsed time: 00h 00m 39s


## Step 5: Inspect the best hyperparameters
- Purpose: retrieve the top-performing configuration.
- Why: this tells us the best units and learning rate found.
- Expected output: printed best values for units and learning rate.

In [ ]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best units: {best_hp.get('units')}")
print(f"Best learning rate: {best_hp.get('learning_rate')}")


Best units: 32
Best learning rate: 0.00046073971733178836


## Step 6: Train the best model
- Purpose: build and train a model using the best hyperparameters.
- Why: final training provides a model ready for evaluation or deployment.
- Expected output: training logs for the best model.

In [ ]:
model = tuner.hypermodel.build(best_hp)
model.fit(x_train, y_train, epochs=20, validation_split=0.2)


Epoch 1/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 445ms/step - loss: 0.1895 - mae: 0.3531 - val_loss: 0.1732 - val_mae: 0.3474
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 0.1566 - mae: 0.3176 - val_loss: 0.1560 - val_mae: 0.3267
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.1592 - mae: 0.3188 - val_loss: 0.1409 - val_mae: 0.3115
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 0.1355 - mae: 0.2915 - val_loss: 0.1286 - val_mae: 0.2978
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.1333 - mae: 0.2870 - val_loss: 0.1187 - val_mae: 0.2850
Epoch 6/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.1117 - mae: 0.2688 - val_loss: 0.1110 - val_mae: 0.2759
Epoch 7/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.1101 - mae: 0.2672 - val_loss: 0.1049 - val_mae: 0.2672
Epoch 8/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.1139 - mae: 0.2724 - val_loss: 0.1005 - val_mae: 0.2594
Epoch 9/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 0.1141 - mae: 0.2790 -